# S09 - Lab exercice 02
## Yago Ramos Sánchez

The attached files are a collection of tweets labelled with sentiment in 3 categories:

sentiments = {
    "LABEL_0": "Bearish", 
    "LABEL_1": "Bullish", 
    "LABEL_2": "Neutral"
}  
Train a LSTM network with the training file. Validate the trained model with the valid file.

In [9]:
# Libraries importation

import torch 
import torch.nn as nn
import pandas as pd
import numpy as np
import re
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix


In [10]:
# Manual seed for reproducibility
torch.manual_seed(42)

# Data loading
train_df= pd.read_csv("sent_train.csv")
valid_df = pd.read_csv("sent_valid.csv")

text_column = "text"
label_column = "label"


In [ ]:
# Text preprocessing

def clean_text(text):
    """Removing URLs, mentions, hashtags, punctuation and converting to lowercase."""
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    text = re.sub(r'#\w+', '', text)  # Remove hashtags
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = text.lower()  # Convert to lowercase
    return text

# Apply cleaning to the text column
train_df[text_column] = train_df[text_column].apply(clean_text)
valid_df[text_column] = valid_df[text_column].apply(clean_text)

# Hyperparameters for text processing
MAX_WORDS = 10000  # Maximum number of words in the vocabulary
MAX_LEN = 100  # Maximum length of input sequences

# Building the vocabulary
all_words = ' '.join(train_df[text_column]).split()
word_counts = Counter(all_words)

# Keep only the most common words
most_common_words = word_counts.most_common(MAX_WORDS - 2)  # Reserve 2 for PAD and UNK
vocab = {word: idx + 2 for idx, (word, _) in enumerate  (most_common_words)}  # Start indexing from 2

def text_to_sequence(text):
    tokens = text.split()
    # Map words to ints, using 1 for unknown words
    sequence = [vocab.get(token, 1) for token in tokens]
    # Truncate if the worg is too long
    if len(sequence) > MAX_LEN:
        sequence = sequence[:MAX_LEN] 
    # Pad with zeros if the sequence is too short
    else:
        sequence += [0] * (MAX_LEN - len(sequence))
    return sequence

